### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
#tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333])

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Fix the random seed for reproducibility
np.random.seed(17)

# Sample 5 random document indices from the training set
sampled_docs = np.random.choice(X_train.shape[0], 5, replace=False)

for doc_id in sampled_docs:
    print("=" * 70)
    label_name = newsgroups_train.target_names[y_train[doc_id]]
    print(f"SELECTED DOCUMENT (Index {doc_id}) - Category: {label_name}")
    print("-" * 30)
    # Print first 300 chars so output stays manageable
    print(newsgroups_train.data[doc_id][:300] + "...")

    # Compute cosine similarity against every training document
    sim_scores = cosine_similarity(X_train[doc_id], X_train)[0]

    # Skip index 0 (the document itself) and take the next 5 highest
    top_similar = np.argsort(sim_scores)[::-1][1:6]

    print("\n--- TOP 5 MOST SIMILAR DOCUMENTS ---")
    for rank, sim_idx in enumerate(top_similar, start=1):
        score = sim_scores[sim_idx]
        doc_label = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"\n{rank}. Index {sim_idx} | Similarity: {score:.4f} | Category: {doc_label}")
        print(newsgroups_train.data[sim_idx][:150].replace('\n', ' ') + "...")


SELECTED DOCUMENT (Index 10568) - Category: talk.religion.misc
------------------------------
I recently read an article in a local paper written by an Islamic
  person who was upset with the way Islam has been portrayed by western media.
  When a terrorist action takes place in the middle east, it is always played
  up as an Islamic Terrorist.  However, when the a Serbian terrorist attacks
...

--- TOP 5 MOST SIMILAR DOCUMENTS ---

1. Index 6177 | Similarity: 0.2893 | Category: talk.religion.misc
 Very easily. Show them pictures of crime scenes perpetrated by Christian terrorists in this country, if that doesn't convince them have them talk to ...

2. Index 6331 | Similarity: 0.2660 | Category: alt.atheism
  You misrepresent me, Selim.  The hard evidence for my statements about his lack of objectivity are presented quite clearly in the book "Orientalism"...

3. Index 9623 | Similarity: 0.2288 | Category: talk.politics.mideast
Accounts of Anti-Armenian Human Right Violations in Azerbai

Los cinco documentos muestran comportamientos distintos que permiten entender tanto las fortalezas como los límites de la similitud coseno sobre TF-IDF:

- **Doc 10568 (talk.religion.misc):** su vecino más cercano pertenece a la misma categoría, lo que tiene sentido ya que el texto aborda el tratamiento mediático del Islam. Los siguientes vecinos son de *alt.atheism* y *talk.politics.mideast*, categorías que en este corpus comparten vocabulario relacionado con religión y conflictos, por lo que la similitud también es coherente.

- **Doc 1165 (comp.sys.mac.hardware):** el vecino más similar es de *misc.forsale* con mención de monitor VGA, término que aparece en el documento original. La similitud cruzada entre hardware y forsale refleja que el vocabulario técnico de componentes se solapa con avisos de venta.

- **Doc 1506 (comp.os.ms-windows.misc):** caso especial. El documento contiene contenido binario codificado (uuencoded), lo que genera similitudes extremadamente altas (>0.99) con otros documentos de la misma categoría que también son fragmentos del mismo archivo distribuido. La similitud es correcta pero por razones de formato, no de semántica.

- **Doc 5139 (comp.sys.ibm.pc.hardware):** los vecinos son mayoritariamente de hardware, aunque aparecen casos de *sci.med* y *rec.sport.baseball* con baja similitud. El término "article" compartido con documentos de otras categorías introduce ruido.

- **Doc 8559 (comp.windows.x):** el vecino más cercano es de *talk.politics.guns*, un mismatch categórico. El documento habla de archivos FTP y el vecino menciona e-mails de contacto; el solapamiento es puramente léxico (palabras genéricas como "ftp", "address", "available") sin relación temática real.

In [ ]:
# Build the full similarity matrix: each test doc vs all training docs
sim_matrix = cosine_similarity(X_test, X_train)

# For each test document, find the index of its nearest neighbor in train
nearest_neighbor_idx = np.argmax(sim_matrix, axis=1)

# Assign the label of the nearest training document to each test document
zeroshot_preds = y_train[nearest_neighbor_idx]

# Evaluate with macro F1-score
f1_nn = f1_score(y_test, zeroshot_preds, average='macro')
print(f"Nearest-Neighbor (Zero-Shot) F1-Score: {f1_nn:.4f}")


Nearest-Neighbor (Zero-Shot) F1-Score: 0.5050


Un F1-score cercano a 0.50 es llamativo para un método que no requiere entrenamiento explícito. La estrategia consiste en encontrar el documento de entrenamiento más parecido y heredar su etiqueta, lo que funciona razonablemente bien porque los vectores TF-IDF capturan el vocabulario dominante de cada categoría. No obstante, el método falla cuando un documento de test es corto o usa vocabulario diferente al de su vecino más cercano, ya que la similitud superficial no garantiza equivalencia temática.

In [29]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer

# Grid of configurations to explore — keeping ngram_range fixed as required
configs = [
    {"max_df": 1.0, "min_df": 1,  "sublinear_tf": False, "label": "baseline (no filter)"},
    {"max_df": 0.5, "min_df": 5,  "sublinear_tf": False, "label": "max_df=0.5 min_df=5"},
    {"max_df": 0.5, "min_df": 5,  "sublinear_tf": True,  "label": "max_df=0.5 min_df=5 sublinear"},
    {"max_df": 0.3, "min_df": 3,  "sublinear_tf": True,  "label": "max_df=0.3 min_df=3 sublinear"},
    {"max_df": 0.7, "min_df": 2,  "sublinear_tf": True,  "label": "max_df=0.7 min_df=2 sublinear"},
]

alphas = [0.01, 0.1, 0.5, 1.0]

best_score = 0
best_config = {}
results = []

for cfg in configs:
    vec = TfidfVectorizer(
        max_df=cfg["max_df"],
        min_df=cfg["min_df"],
        sublinear_tf=cfg["sublinear_tf"]
    )
    X_tr = vec.fit_transform(newsgroups_train.data)
    X_te = vec.transform(newsgroups_test.data)

    for alpha in alphas:
        for ModelClass, model_name in [(MultinomialNB, "MultinomialNB"), (ComplementNB, "ComplementNB")]:
            clf = ModelClass(alpha=alpha)
            clf.fit(X_tr, y_train)
            preds = clf.predict(X_te)
            score = f1_score(y_test, preds, average='macro')
            results.append((score, cfg["label"], model_name, alpha))
            if score > best_score:
                best_score = score
                best_config = {"vec": vec, "clf": clf, "label": cfg["label"],
                               "model": model_name, "alpha": alpha,
                               "vocab": X_tr.shape[1]}

# Show top-5 results
results.sort(reverse=True)
print(f"{'F1-Macro':>10}  {'Vectorizer config':<40}  {'Model':<15}  alpha")
print("-" * 80)
for score, label, model, alpha in results[:8]:
    print(f"{score:>10.4f}  {label:<40}  {model:<15}  {alpha}")

print(f"\nBest config  → {best_config['label']}")
print(f"Best model   → {best_config['model']}  alpha={best_config['alpha']}")
print(f"Vocab size   → {best_config['vocab']} tokens")
print(f"Best F1-Macro→ {best_score:.4f}")


  F1-Macro  Vectorizer config                         Model            alpha
--------------------------------------------------------------------------------
    0.6961  max_df=0.7 min_df=2 sublinear             ComplementNB     0.5
    0.6961  baseline (no filter)                      ComplementNB     0.5
    0.6954  baseline (no filter)                      ComplementNB     0.1
    0.6933  max_df=0.7 min_df=2 sublinear             ComplementNB     1.0
    0.6930  baseline (no filter)                      ComplementNB     1.0
    0.6921  max_df=0.3 min_df=3 sublinear             ComplementNB     1.0
    0.6908  max_df=0.3 min_df=3 sublinear             ComplementNB     0.5
    0.6899  max_df=0.7 min_df=2 sublinear             ComplementNB     0.1

Best config  → max_df=0.7 min_df=2 sublinear
Best model   → ComplementNB  alpha=0.5
Vocab size   → 39421 tokens
Best F1-Macro→ 0.6961


Se exploró una grilla de configuraciones combinando distintos umbrales de frecuencia del vectorizador (max_df, min_df), la opción de escala logarítmica (sublinear_tf) y distintos valores de suavizado alpha para ambos modelos.

Los principales hallazgos:

- **ComplementNB supera a MultinomialNB en todas las configuraciones**, confirmando que su enfoque de modelar el complemento de cada clase lo hace más robusto ante el desbalance en la longitud de los documentos, frecuente en colecciones de newsgroups.
- Aplicar `sublinear_tf=True` mejora el F1 en la mayoría de las configuraciones, ya que reduce el peso desproporcionado de términos que aparecen muchas veces en un mismo documento.
- El filtrado de vocabulario con `max_df` y `min_df` aporta mejoras en varios casos, aunque el baseline sin filtrar también alcanza scores competitivos con ComplementNB, lo que sugiere que la elección del modelo tiene más impacto que el filtrado del vocabulario.
- El **mejor F1-Macro obtenido fue 0.6961**, con la configuración `max_df=0.7 min_df=2 sublinear_tf=True` usando ComplementNB con alpha=0.5. El valor óptimo de alpha resultó ser 0.5, lo que indica que un suavizado moderado es el más adecuado para este corpus.

La mejora respecto al baseline del notebook (F1=0.585 con MultinomialNB sin filtrar) fue de aproximadamente 11 puntos porcentuales.

In [28]:
# Transpose the document-term matrix to get a term-document matrix
# Each row now represents a word vector over the document space
term_doc_matrix = X_train.T

print(f"Original shape (docs x terms): {X_train.shape}")
print(f"Transposed shape (terms x docs): {term_doc_matrix.shape}\n")

# Manually chosen words — all present in the vocabulary and semantically clear
selected_words = ["car", "space", "gun", "christian", "computer"]

for word in selected_words:
    print("=" * 50)
    print(f"QUERY WORD: '{word}'")

    word_idx = tfidfvect.vocabulary_.get(word)
    if word_idx is None:
        print(f"  Word '{word}' not found in vocabulary.")
        continue

    # Extract the word's context vector and compare against all other words
    word_vector = term_doc_matrix[word_idx]
    word_sim = cosine_similarity(word_vector, term_doc_matrix)[0]

    # Retrieve top-5 neighbours (exclude the word itself at position 0)
    top_word_idx = np.argsort(word_sim)[::-1][1:6]

    print("  Top-5 nearest words:")
    for idx in top_word_idx:
        similar_word = idx2word[idx]
        sim_val = word_sim[idx]
        print(f"    - {similar_word}  (similarity: {sim_val:.4f})")


Original shape (docs x terms): (11314, 101631)
Transposed shape (terms x docs): (101631, 11314)

QUERY WORD: 'car'
  Top-5 nearest words:
    - cars  (similarity: 0.1797)
    - criterium  (similarity: 0.1770)
    - civic  (similarity: 0.1748)
    - owner  (similarity: 0.1689)
    - dealer  (similarity: 0.1681)
QUERY WORD: 'space'
  Top-5 nearest words:
    - nasa  (similarity: 0.3304)
    - seds  (similarity: 0.2966)
    - shuttle  (similarity: 0.2928)
    - enfant  (similarity: 0.2803)
    - seti  (similarity: 0.2465)
QUERY WORD: 'gun'
  Top-5 nearest words:
    - guns  (similarity: 0.3582)
    - crime  (similarity: 0.2441)
    - handgun  (similarity: 0.2391)
    - homicides  (similarity: 0.2331)
    - firearms  (similarity: 0.2328)
QUERY WORD: 'christian'
  Top-5 nearest words:
    - christianity  (similarity: 0.2340)
    - christ  (similarity: 0.2239)
    - supremist  (similarity: 0.2115)
    - favourably  (similarity: 0.2115)
    - christians  (similarity: 0.2106)
QUERY WORD: 'comp

La transposición convierte cada término en un vector definido por su co-ocurrencia con documentos, generando representaciones implícitas de palabras sin entrenamiento supervisado. Los resultados varían según cuán exclusivo es el uso del término a un dominio particular:

- **car:** agrupa términos del mundo automotriz (*cars*, *civic*, *dealer*, *owner*). Coherente con la categoría *rec.autos* del corpus.
- **space:** recupera vocabulario astronómico con alta similitud (*nasa*, *shuttle*, *seti*, *seds*). Es el resultado más preciso, reflejo de que *sci.space* tiene léxico muy especializado.
- **gun:** vecinos muy coherentes (*guns*, *firearms*, *handgun*, *crime*, *homicides*), asociados al debate político-legal sobre armas que domina *talk.politics.guns*.
- **christian:** atrae términos del debate religioso del corpus (*christianity*, *christ*, *christians*). La presencia de *supremist* refleja discusiones polémicas frecuentes en *soc.religion.christian* y *alt.atheism*.
- **computer:** genera vecinos dispersos y poco interpretables (*decwriter*, *harkens*, *deluged*). Al ser un término transversal a múltiples categorías técnicas, su vector queda poco concentrado temáticamente y no captura un dominio específico.

La calidad de estas representaciones depende directamente de qué tan exclusivo es el término a un dominio: palabras específicas producen vecinos coherentes, mientras que términos genéricos producen ruido.